# Pilot — industrial equipment extraction from PDFs

> This notebook is an experimental pilot. It does not attempt to process the entire manual. It selects 5–10 representative pages, builds manual ground truth, compares native text extraction, rules, table extraction, OCR, and LLMs, then identifies the best extraction strategy for each field before the full pipeline is built.

No value is invented: any information not visible in the document remains `None`. i-Sense fields are handled in a separate mapping layer.

## 1. Configuration

Set the PDF path and choose five to ten representative pages after an initial inspection. Page numbers are **1-based** (as in a PDF reader).

In [14]:
import os
from pathlib import Path
import json, re, random, warnings
from collections import defaultdict

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR     = PROJECT_ROOT / 'data' / 'raw'

PDF_AUSTCOLD = DATA_DIR / 'AUSTCOLD.pdf'
PDF_MYCOM    = DATA_DIR / 'MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf'

REPRESENTATIVE_PAGES = [
    {'page': 3, 'type': 'summary', 'description': 'Summary'},
    {'page': 9, 'type': 'section', 'description': 'Scanned/image-heavy page'},
    {'page': 18, 'type': 'safety_instructions', 'description': 'Safety instructions'},
    {'page': 28, 'type': 'maintenance_log', 'description': 'Plant history and maintenance work log'},
    {'page': 38, 'type': 'utilities_consumption_schedule', 'description': 'Utilities consumption and equipment power schedule'},
    {'page': 43, 'type': 'document_metadata', 'description': 'Document metadata and revision history'},
    {'page': 44, 'type': 'cause_effect_diagram', 'description': 'Refrigeration system cause and effect diagram'},
    {'page': 51, 'type': 'equipment_overview', 'description': 'Refrigeration system equipment overview and scope of supply'},
    {'page': 192, 'type': 'technical_data_sheet', 'description': 'Technical data sheet'},
    {'page': 189, 'type': 'drawing', 'description': 'Technical drawing'},
]

OUTPUT_DIR = Path('../results')
ENABLE_OCR = True
ENABLE_LLM = False
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Résultats : {OUTPUT_DIR.resolve()}')

Résultats : C:\Users\intel\Downloads\industrial-equipment-data-extraction\results


In [15]:
PDF_PATH = PDF_AUSTCOLD

## 2. Dependencies and PDF loading

PyMuPDF is the main component. OCR, table extraction, and LLMs are optional: an unavailable dependency is reported and the remaining experiments continue.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def optional_import(name):
    try:
        return __import__(name), None
    except ImportError as exc:
        return None, str(exc)

fitz, FITZ_ERROR = optional_import('fitz')
pytesseract, OCR_ERROR = optional_import('pytesseract')
pdfplumber, PDFPLUMBER_ERROR = optional_import('pdfplumber')
camelot, CAMELOT_ERROR = optional_import('camelot')

doc = None
if fitz is not None and PDF_PATH.exists():
    doc = fitz.open(PDF_PATH)
    print(f'PDF chargé : {PDF_PATH.name} ({doc.page_count:,} pages)')
else:
    print('Configurez PDF_PATH vers un PDF existant avant les expériences d’extraction.')

PDF chargé : AUSTCOLD.pdf (2,870 pages)


## 3. Page selection and visualisation

Examples should represent different document types. This cell displays the image, metadata, and a native-text preview without analysing the entire manual.

In [17]:
def require_pdf():
    if doc is None:
        raise RuntimeError('PDF non chargé : configurez PDF_PATH et installez PyMuPDF.')

def render_page(page_number, dpi=150):
    require_pdf()
    pix = doc[page_number - 1].get_pixmap(dpi=dpi, alpha=False)
    return pix.pil_image()

def extract_native_text(pdf_path, page_number):
    require_pdf()
    text = doc[page_number - 1].get_text('text')
    return {'text': text, 'page': page_number, 'method': 'native_text'}

def extract_words_with_coordinates(pdf_path, page_number):
    require_pdf()
    words = doc[page_number - 1].get_text('words')
    return pd.DataFrame(words, columns=['x0','y0','x1','y1','word','block','line','word_no'])

def show_selected_pages(show_native_text=True):
    for item in REPRESENTATIVE_PAGES:
        page = item['page']
        print(f"Page {page} | {item.get('type','unknown')} | {item.get('description','')}")
        display(render_page(page))
        if show_native_text:
            print(extract_native_text(PDF_PATH, page)['text'][:1500] or '[texte natif vide]')

# show_selected_pages()

## 4. Manual ground truth

Complete this only after visual inspection. Every value must be explicitly visible on the stated page; otherwise use `None`. This small pilot dataset is a decision tool, not a final statistical validation.

In [18]:
GROUND_TRUTH = {
    189: {
        'family': None,
        'asset_name': 'Rotor',
        'reference': 'C4362',
        'power': None,
        'outlier': None,
        'manufacturer': 'CoFimco',
        'asset_diagram': True,
    },

    192: {
        'family': '17',
        'asset_name': 'Three-phase Induction Motor - Squirrel Cage',
        'reference': '175132/2011',
        'power': '30 kW',
        'outlier': None,
        'manufacturer': 'WEG Indústrias S.A.',
        'asset_diagram': False,
    },
}

FIELDS = ['family', 'asset_name', 'reference', 'power', 'outlier', 'manufacturer', 'asset_diagram']
ground_truth_df = pd.DataFrame.from_dict(GROUND_TRUTH, orient='index').rename_axis('page').reset_index()
display(ground_truth_df)
ground_truth_df.to_csv(OUTPUT_DIR / 'ground_truth.csv', index=False)

,page,family,asset_name,reference,power,outlier,manufacturer,asset_diagram
0,189,NaN,Rotor,C4362,NaN,None,CoFimco,True
1,192,17,Three-phase Induction Motor - Squirrel Cage,175132/2011,30 kW,None,WEG Indústrias S.A.,False


## 5. Native extraction and quality check

Native text is the low-cost baseline. OCR is considered only when native text is insufficient or the page is clearly image-based/scanned.

In [19]:
def text_quality(text):
    clean = re.sub(r'\s+', ' ', text or '').strip()
    alnum = sum(c.isalnum() for c in clean)
    return {'characters': len(clean), 'alnum_ratio': alnum / max(len(clean), 1), 'sufficient': len(clean) >= 40 and alnum / max(len(clean), 1) >= 0.35}

def needs_ocr(text, page_image=None):
    quality = text_quality(text)
    return not quality['sufficient']

def native_page_result(page):
    native = extract_native_text(PDF_PATH, page)
    native['quality'] = text_quality(native['text'])
    native['needs_ocr'] = needs_ocr(native['text'])
    return native

[native_page_result(x['page']) for x in REPRESENTATIVE_PAGES]

[{'text': 'INTESCA \n \nOPERATING & MAINTENANCE INSTRUCTION INDEX \n \nSECTION \n1) \nSERVICE CONTACT DETAILS \n1.01 \nContact details \n1.02 \nWarning & material safety data sheets \n1.03 \nSafety precautions \n1.04 \nPackage warranty and support information \n1.05 \nPlant history log \n1.06 \nOil consumption log \n1.07 \nUtilities consumption schedule \n1.08 \nCause & effect chart \n \n \n \n2)  \nDESCRIPTIONS \n2.01 \nControl Philosophy \n \n \n \n3) \nTECHNICAL DATA \n3.01 \nTechnical schedule \n3.02 \nCompressor data sheet \n3.03 \nCompressor motor data sheet \n3.04 \nOil pump data sheet \n3.05 \nOil pump motor data sheet \n3.06 \nOil cooler data sheet  \n3.07 \nOil separator data sheet \n3.08 \nLiquid receiver data sheet \n3.09 \nPurger data sheet \n3.10 \nEconomiser data sheet \n3.11 \nCondenser data sheet \n3.12 \nCondenser fan data sheet \n3.13 \nCondenser fan motor data sheet \n3.14 \nCondenser vibration switch data sheet \n3.15 \nFinal stage oil separator data sheet \n3.16 \

## 6. Regex baseline

Ces règles sont volontairement courtes et lisibles. Elles constituent une baseline mesurable, non un moteur métier complet. Modifier les motifs uniquement après avoir observé les conventions réelles du manuel.

In [20]:
POWER_RE = re.compile(
    r'(?i)\b(?:power|motor\s*power|rated\s*power|output|puissance)'
    r'\s*[:=]?\s*(\d+(?:[.,]\d+)?)\s*(kW|MW|W|HP)\b'
)

LABELED_RE = {
    'reference': re.compile(
        r'(?im)^\s*'
        r'(?:tag|reference|ref\.?|equipment\s*no\.?|asset\s*no\.?)'
        r'\s*[:#=]\s*([^\n]+)'
    ),

    'manufacturer': re.compile(
        r'(?im)^\s*'
        r'(?:manufacturer|maker|fabricant)'
        r'\s*[:#=]\s*([^\n]+)'
    ),

    'asset_name': re.compile(
        r'(?im)^\s*'
        r'(?:equipment\s*(?:name|designation)|asset\s*name|designation)'
        r'\s*[:#=]\s*([^\n]+)'
    ),
}

In [21]:
def _first_match(pattern, text, group=1):
    match = pattern.search(text or '')
    return match.group(group).strip() if match else None

# 2. Individual extraction functions
def extract_reference(text):
    return _first_match(LABELED_RE['reference'], text)


def extract_manufacturer(text):
    return _first_match(LABELED_RE['manufacturer'], text)


def extract_asset_name(text):
    return _first_match(LABELED_RE['asset_name'], text)


def extract_power(text):
    match = POWER_RE.search(text or '')

    if not match:
        return {
            'power': None,
            'power_unit': None
        }

    return {
        'power': float(match.group(1).replace(',', '.')),
        'power_unit': match.group(2)
    }

# 3. Fields that cannot currently be reliably extracted
def extract_family(text):
    """
    Family is an i-Sense classification.
    It should not be guessed from the PDF text alone.
    """
    return None


def extract_outlier(text):
    """
    No reliable Outlier pattern has been established yet.
    """
    return None


def extract_asset_diagram(text):
    """
    A text-only extractor cannot reliably determine whether
    a page contains an asset diagram.
    This should be determined from page type / visual analysis.
    """
    return None

# 4. Complete rule-based extraction
def extract_rules(text):
    
    power = extract_power(text)

    values = {
        'family': extract_family(text),
        'asset_name': extract_asset_name(text),
        'reference': extract_reference(text),
        'power': (
            f"{power['power']} {power['power_unit']}"
            if power['power'] is not None
            else None
        ),
        'outlier': extract_outlier(text),
        'manufacturer': extract_manufacturer(text),
        'asset_diagram': extract_asset_diagram(text),
    }

    return {
        'method': 'rules',
        'values': values
    }

In [22]:
extract_rules(extract_native_text(PDF_PATH, REPRESENTATIVE_PAGES[8]['page'])['text'])

{'method': 'rules',
 'values': {'family': None,
  'asset_name': None,
  'reference': None,
  'power': '30.0 kW',
  'outlier': None,
  'manufacturer': None,
  'asset_diagram': None}}

## 8. OCR fallback ciblé

La page n’est envoyée à OCR que lorsque la qualité du texte natif échoue. Si Tesseract ou son exécutable n’est pas configuré, le résultat conserve un message et l’expérience se poursuit.

In [23]:
import pytesseract

print("pytesseract:", pytesseract.get_tesseract_version())

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [ ]:
def ocr_page(pdf_path, page_number, dpi=300):
    result = {'text': '', 'method': 'ocr', 'page': page_number, 'available': False, 'message': ''}
    if not ENABLE_OCR:
        result['message'] = 'OCR désactivé dans la configuration.'; return result
    if pytesseract is None:
        result['message'] = f'Tesseract indisponible : {OCR_ERROR}'; return result
    try:
        result['text'] = pytesseract.image_to_string(render_page(page_number, dpi=dpi))
        result['available'] = True
    except Exception as exc:
        result['message'] = f'Échec OCR : {exc}'
    return result

def ocr_fallback(native_result):
    return ocr_page(PDF_PATH, native_result['page']) if native_result['needs_ocr'] else {'page': native_result['page'], 'method': 'ocr', 'skipped': True, 'message': 'Texte natif suffisant.'}

ocr_fallback(extract_native_text(PDF_PATH, REPRESENTATIVE_PAGES[2]['page'])['text'])

TypeError: string indices must be integers, not 'str'

## 9. LLM — uniquement pour les cas difficiles

Par défaut le LLM n’est pas appelé. Le payload ne contient que le texte, l’OCR et les tableaux de la page ciblée. L’implémentation du client est laissée explicite car elle dépend du fournisseur et des identifiants approuvés.

In [ ]:
LLM_SCHEMA = {'equipment_name': None, 'tag_reference': None, 'manufacturer': None, 'model': None, 'serial_number': None, 'power': None, 'power_unit': None, 'source_page': None, 'confidence': {}}
LLM_INSTRUCTIONS = '''Extrais uniquement les valeurs explicitement présentes dans les sources fournies. Si une valeur est absente, retourne null. N’infère jamais Family, Class, Structure, Group ou Entity i-Sense. Retourne la page source et une confiance par champ.'''

def extract_with_llm(native_result, ocr_result, table_result):
    payload = {'instructions': LLM_INSTRUCTIONS, 'expected_schema': LLM_SCHEMA, 'native_text': native_result['text'], 'ocr_text': ocr_result.get('text'), 'tables': [t.fillna('').to_dict('records') for t in table_result.get('tables', [])]}
    if not ENABLE_LLM:
        return {'method': 'llm', 'called': False, 'message': 'LLM désactivé', 'payload': payload}
    # Brancher ici un client à sortie JSON structurée approuvé. Ne pas envoyer le PDF entier.
    return {'method': 'llm', 'called': False, 'message': 'Client LLM non configuré', 'payload': payload}


## 10. Pipeline expérimental par page

Chaque résultat reste séparé par méthode. Le pipeline ne fusionne pas silencieusement les résultats : la comparaison doit révéler quelle méthode fonctionne réellement.

In [ ]:
def run_page_experiment(pdf_path, page_number):
    native = native_page_result(page_number)
    rules = extract_rules(native['text'])
    table = extract_tables(pdf_path, page_number)
    ocr = ocr_fallback(native)
    llm = extract_with_llm(native, ocr, table)
    return {'page': page_number, 'native_text': native, 'rules': rules, 'tables': table, 'ocr': ocr, 'llm': llm}

EXPERIMENTS = {item['page']: run_page_experiment(PDF_PATH, item['page']) for item in REPRESENTATIVE_PAGES} if doc else {}
print(f'{len(EXPERIMENTS)} page(s) expérimentée(s)')
with open(OUTPUT_DIR / 'raw_experiments.json', 'w', encoding='utf-8') as handle:
    json.dump(EXPERIMENTS, handle, default=str, ensure_ascii=False, indent=2)

NameError: name 'extract_tables' is not defined

## 11. Comparaison avec le ground truth

La normalisation tolère seulement casse, espaces et format d’unité évident. Elle ne rend pas égales des valeurs différentes, par exemple `15 kW` et `18 kW`.

In [ ]:
def normalize_value(value, field=None):
    if value is None or (isinstance(value, float) and np.isnan(value)): return None
    value = re.sub(r'\s+', ' ', str(value).strip()).casefold()
    if field == 'power_unit': return value.replace('kw', 'kw').replace(' ', '')
    if field == 'power':
        number = re.search(r'\d+(?:[.,]\d+)?', value)
        return str(float(number.group().replace(',', '.'))) if number else value
    return value

def compare_to_ground_truth(prediction, truth, method):
    rows = []
    values = prediction.get('values', prediction)
    for field in FIELDS:
        expected, observed = truth.get(field), values.get(field)
        correct = normalize_value(expected, field) == normalize_value(observed, field) if expected is not None else observed is None
        rows.append({'page': truth.get('source_page'), 'field': field, 'ground_truth': expected, 'prediction': observed, 'correct': correct, 'method': method})
    return rows

def method_predictions(experiment):
    native = {'source_page': experiment['page']}  # texte brut : évaluable manuellement, pas une valeur structurée
    return {'rules': experiment['rules'], 'ocr': {}, 'table': {}, 'llm': {}}

comparison_rows = []
for page, truth in GROUND_TRUTH.items():
    if page in EXPERIMENTS:
        for method, prediction in method_predictions(EXPERIMENTS[page]).items():
            comparison_rows.extend(compare_to_ground_truth(prediction, truth, method))
field_comparison = pd.DataFrame(comparison_rows)
display(field_comparison)
field_comparison.to_csv(OUTPUT_DIR / 'field_comparison.csv', index=False)

""


## 12. Métriques, erreurs et méthode gagnante

Les métriques sont descriptives sur un petit échantillon pilote. Une méthode n’est recommandée pour un champ que si elle a produit des valeurs évaluables et correctes ; sinon le résultat reste `UNRESOLVED`.

In [ ]:
def classify_error(row):
    if pd.isna(row['prediction']) or row['prediction'] is None: return 'MISSING'
    if row['field'] == 'power_unit': return 'WRONG_UNIT'
    return 'WRONG_VALUE'

if not field_comparison.empty:
    field_comparison['status'] = np.where(field_comparison['correct'], 'CORRECT', 'INCORRECT')
    errors = field_comparison.loc[~field_comparison['correct']].copy()
    errors['error_type'] = errors.apply(classify_error, axis=1)
    summary = (field_comparison.assign(found=field_comparison['prediction'].notna())
               .groupby(['field','method'], as_index=False)
               .agg(values_found=('found','sum'), correct=('correct','sum'), total=('correct','size')))
    summary['coverage'] = summary['values_found'] / summary['total']
    summary['accuracy'] = summary['correct'] / summary['total']
    best_method_by_field = (summary.sort_values(['field','accuracy','coverage'], ascending=[True,False,False])
                            .groupby('field', as_index=False).first()[['field','method','accuracy','coverage']]
                            .rename(columns={'method':'best_method'}))
else:
    errors = pd.DataFrame(columns=['page','field','ground_truth','prediction','method','error_type'])
    summary = pd.DataFrame(columns=['field','method','values_found','correct','total','coverage','accuracy'])
    best_method_by_field = pd.DataFrame({'field': FIELDS, 'best_method': 'UNRESOLVED'})
display(summary); display(errors); display(best_method_by_field)
errors.to_csv(OUTPUT_DIR / 'error_analysis.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'method_summary.csv', index=False)

,field,method,values_found,correct,total,coverage,accuracy


,page,field,ground_truth,prediction,method,error_type


,field,best_method
0,equipment_name,UNRESOLVED
1,tag_reference,UNRESOLVED
2,manufacturer,UNRESOLVED
3,model,UNRESOLVED
4,serial_number,UNRESOLVED
5,power,UNRESOLVED
6,power_unit,UNRESOLVED
7,source_page,UNRESOLVED


## 13. Mapping i-Sense — couche séparée

Les règles métier ne doivent pas se trouver dans l’extraction documentaire. Le fichier externe ci-dessous est un squelette : les valeurs d’exemple ne sont pas des règles validées et ne doivent pas être utilisées comme telles.

In [ ]:
MAPPING_PATH = OUTPUT_DIR / 'isense_mapping.csv'
if not MAPPING_PATH.exists():
    pd.DataFrame(columns=['normalized_equipment_type','Family','Class','Structure','Group']).to_csv(MAPPING_PATH, index=False)
mapping_df = pd.read_csv(MAPPING_PATH)

def map_to_isense(equipment_type, mapping_df):
    if equipment_type is None or mapping_df.empty: return {'status': 'UNRESOLVED'}
    candidates = mapping_df[mapping_df['normalized_equipment_type'].astype(str).str.casefold() == str(equipment_type).casefold()]
    return candidates.iloc[0].to_dict() if len(candidates) == 1 else {'status': 'UNRESOLVED'}

display(mapping_df)

,normalized_equipment_type,Family,Class,Structure,Group


## 14. Review workflow

Une valeur est acceptée uniquement si elle est sourcée, non conflictuelle et suffisamment fiable. Les valeurs requises absentes, les conflits, l’OCR/LLM peu fiable et le mapping non résolu vont en revue.

In [ ]:
REQUIRED_FIELDS = {'equipment_name','tag_reference'}
def needs_review(record):
    reasons = []
    if not record.get('source_page'): reasons.append('source page missing')
    for field in REQUIRED_FIELDS:
        if not record.get(field): reasons.append(f'required field missing: {field}')
    if record.get('conflicting_values'): reasons.append('conflicting values')
    if record.get('low_confidence'): reasons.append('low confidence')
    if record.get('mapping_status') == 'UNRESOLVED': reasons.append('i-Sense mapping unresolved')
    return ('REVIEW' if reasons else 'ACCEPT'), reasons

review_queue = []
for page, truth in GROUND_TRUTH.items():
    status, reasons = needs_review(truth)
    review_queue.append({'page': page, 'field': '*', 'value': '', 'status': status, 'reason': '; '.join(reasons)})
review_queue = pd.DataFrame(review_queue)
display(review_queue)
review_queue.to_csv(OUTPUT_DIR / 'review_queue.csv', index=False)

""


## 15. Visualisations utiles

Les graphiques restent limités à ce qui aide à décider : précision par champ/méthode, erreurs par méthode, et besoin OCR/LLM.

In [ ]:
if not summary.empty:
    pivot = summary.pivot(index='field', columns='method', values='accuracy').fillna(0)
    pivot.plot(kind='bar', figsize=(11,4), ylabel='Accuracy', title='Accuracy par champ et méthode')
    plt.ylim(0, 1); plt.tight_layout(); plt.show()
    errors.groupby('method').size().plot(kind='bar', title='Nombre d’erreurs par méthode', ylabel='Erreurs')
    plt.tight_layout(); plt.show()
ocr_needed = sum(exp['native_text']['needs_ocr'] for exp in EXPERIMENTS.values())
print(f'Pages nécessitant potentiellement OCR : {ocr_needed}/{len(EXPERIMENTS)}')
print(f'Cas LLM réellement exécutés : {sum(bool(exp["llm"].get("called")) for exp in EXPERIMENTS.values())}')

Pages nécessitant potentiellement OCR : 3/10
Cas LLM réellement exécutés : 0


## 16. Résumé automatique et conclusion expérimentale

Cette recommandation est une hypothèse issue du pilote. Elle doit être confirmée par un échantillon plus large avant tout déploiement.

In [ ]:
print('PILOT SUMMARY')
print(f'Pages evaluated: {len(EXPERIMENTS)}')
if not summary.empty:
    for method, group in summary.groupby('method'):
        print(f'{method}: coverage={group.coverage.mean():.0%}, accuracy={group.accuracy.mean():.0%}')
    print('\nBest method by field:')
    display(best_method_by_field)
else:
    print('Complétez GROUND_TRUTH et exécutez les expériences pour obtenir les métriques.')
print('\nRECOMMENDED PILOT ARCHITECTURE')
print('1. Texte PDF natif en première couche')
print('2. Rules/regex pour valeurs structurées validées')
print('3. Table extraction pour pages à tableaux qui réussissent le test')
print('4. OCR seulement si texte natif insuffisant')
print('5. LLM seulement pour cas difficiles et ambigus, sur pages ciblées')
print('6. Mapping métier i-Sense séparé')
print('7. Revue humaine pour données manquantes, conflictuelles ou non sourcées')

# Résultat consolidé minimal exportable (à alimenter à partir de la méthode gagnante validée).
pd.DataFrame([{'page': p, 'native_text_chars': len(e['native_text']['text']), 'ocr_needed': e['native_text']['needs_ocr']} for p,e in EXPERIMENTS.items()]).to_csv(OUTPUT_DIR / 'extraction_results.csv', index=False)

PILOT SUMMARY
Pages evaluated: 10
Complétez GROUND_TRUTH et exécutez les expériences pour obtenir les métriques.

RECOMMENDED PILOT ARCHITECTURE
1. Texte PDF natif en première couche
2. Rules/regex pour valeurs structurées validées
3. Table extraction pour pages à tableaux qui réussissent le test
4. OCR seulement si texte natif insuffisant
5. LLM seulement pour cas difficiles et ambigus, sur pages ciblées
6. Mapping métier i-Sense séparé
7. Revue humaine pour données manquantes, conflictuelles ou non sourcées


## NEXT ACTION

1. Review the representative pages.
2. Fill the ground truth manually.
3. Run the experiments.
4. Review field-level errors.
5. Validate the winning method for each field.
6. Complete the i-Sense mapping table.
7. Only then design the full extraction architecture.